# 相关分析完整教程：从散点图到 Pearson 相关系数

## 📚 教学目标
1. 理解 Pearson 相关系数 $r$ 的含义及其取值范围
2. 掌握 $r$ 的两种计算公式（原始数据版 & z 分数版）
3. 理解相关性的假设检验（$t$ 检验）
4. 区分相关性和因果性——"相关不等于因果"
5. 识别伪相关和可解释变异（$r^2$）

**参考书**：《基础统计学(第14版)》（Triola）第 10 章 §10-1

**教学策略**：先用强力球彩票的 9 组数据手算每一步，再用 scipy 验证，最后扩展到大样本模拟

## 1. 场景设定：强力球彩票头奖与销售量

### 🎯 问题
强力球（Powerball）彩票的头奖金额越高，彩票销售量是否也越高？
如果我们能用一个**数值**来度量这两个变量之间的**线性关系强度**，就能回答这个问题。

### 📖 书中的数据
> 下表列出了强力球彩票 9 次开奖的头奖金额（百万美元）和对应的销售量（百万张）：
>
> | 头奖（百万$） | 334 | 127 | 300 | 227 | 202 | 180 | 164 | 145 | 255 |
> |---|---|---|---|---|---|---|---|---|---|
> | 销售（百万张） | 54 | 16 | 41 | 27 | 23 | 18 | 18 | 16 | 26 |

我们将在本教程中一步步计算 Pearson 相关系数 $r$，验证书中给出的结果 $r = 0.947$。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置中文字体和样式
import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

## 2. 散点图的绘制与解读

散点图是相关分析的**第一步**。通过散点图可以直观判断两个变量之间是否存在线性关系、关系的方向和强度。

### 📐 散点图的四种典型形态

| 形态 | $r$ 值 | 含义 |
|------|--------|------|
| (a) 线性正相关 | $r$ 接近 +1 | $x$ 增大时 $y$ 也增大 |
| (b) 线性负相关 | $r$ 接近 -1 | $x$ 增大时 $y$ 减小 |
| (c) 无相关 | $r$ 接近 0 | $x$ 与 $y$ 无线性关系 |
| (d) 非线性关系 | $r$ 中等 | 存在关系但不是直线 |

In [ ]:
# ========== 散点图四种典型形态 ==========
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

n_demo = 50

# (a) 线性正相关
x_a = np.random.normal(0, 1, n_demo)
y_a = 0.9 * x_a + np.random.normal(0, 0.4, n_demo)
r_a, _ = stats.pearsonr(x_a, y_a)
axes[0, 0].scatter(x_a, y_a, color='#2ecc71', alpha=0.7, edgecolors='white', s=60)
axes[0, 0].set_title(f'(a) Linear Positive: r = {r_a:.2f}', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('X', fontsize=12)
axes[0, 0].set_ylabel('Y', fontsize=12)
axes[0, 0].grid(alpha=0.3)

# (b) 线性负相关
x_b = np.random.normal(0, 1, n_demo)
y_b = -0.9 * x_b + np.random.normal(0, 0.4, n_demo)
r_b, _ = stats.pearsonr(x_b, y_b)
axes[0, 1].scatter(x_b, y_b, color='#e74c3c', alpha=0.7, edgecolors='white', s=60)
axes[0, 1].set_title(f'(b) Linear Negative: r = {r_b:.2f}', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('X', fontsize=12)
axes[0, 1].set_ylabel('Y', fontsize=12)
axes[0, 1].grid(alpha=0.3)

# (c) 无相关
x_c = np.random.normal(0, 1, n_demo)
y_c = np.random.normal(0, 1, n_demo)
r_c, _ = stats.pearsonr(x_c, y_c)
axes[1, 0].scatter(x_c, y_c, color='steelblue', alpha=0.7, edgecolors='white', s=60)
axes[1, 0].set_title(f'(c) No Correlation: r = {r_c:.2f}', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('X', fontsize=12)
axes[1, 0].set_ylabel('Y', fontsize=12)
axes[1, 0].grid(alpha=0.3)

# (d) 非线性关系
x_d = np.linspace(-2, 2, n_demo)
y_d = x_d ** 2 + np.random.normal(0, 0.3, n_demo)
r_d, _ = stats.pearsonr(x_d, y_d)
axes[1, 1].scatter(x_d, y_d, color='#e67e22', alpha=0.7, edgecolors='white', s=60)
axes[1, 1].set_title(f'(d) Non-linear: r = {r_d:.2f}', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('X', fontsize=12)
axes[1, 1].set_ylabel('Y', fontsize=12)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  图(a)：r 接近 +1，x 增大时 y 也增大，点紧密围绕一条上升直线")
print("  图(b)：r 接近 -1，x 增大时 y 减小，点紧密围绕一条下降直线")
print("  图(c)：r 接近 0，x 与 y 之间没有线性关系，点呈随机散布")
print("  图(d)：r 中等偏低，但存在明显的非线性（抛物线）关系")
print("\n⚠️ 注意：r 只度量线性关系！非线性关系可能被 r 低估。")

## 3. Pearson 相关系数 $r$ 的计算公式

### 📐 公式 10-1（原始数据版）

$$r = \frac{n\sum xy - (\sum x)(\sum y)}{\sqrt{n\sum x^2 - (\sum x)^2} \cdot \sqrt{n\sum y^2 - (\sum y)^2}}$$

其中：
- $n$ = 数据对的数量
- $\sum x$ = 所有 $x$ 值的总和
- $\sum y$ = 所有 $y$ 值的总和
- $\sum xy$ = 所有 $x \cdot y$ 乘积的总和
- $\sum x^2$ = 所有 $x^2$ 的总和
- $\sum y^2$ = 所有 $y^2$ 的总和

### 📐 公式 10-2（z 分数版）

$$r = \frac{\sum (z_x \cdot z_y)}{n - 1}$$

其中：
- $z_x = \dfrac{x - \bar{x}}{s_x}$（$x$ 的 z 分数）
- $z_y = \dfrac{y - \bar{y}}{s_y}$（$y$ 的 z 分数）
- $\bar{x}, \bar{y}$ = 样本均值
- $s_x, s_y$ = 样本标准差

### 📖 $r$ 的性质

| 性质 | 说明 |
|------|------|
| $-1 \leq r \leq 1$ | 取值范围有界 |
| 不受单位变化影响 | 将 $x$ 从"米"改为"厘米"，$r$ 不变 |
| 交换 $x, y$ 不影响 $r$ | $r_{xy} = r_{yx}$ |
| 仅度量线性关系 | 抛物线等非线性关系可能 $r \approx 0$ |
| 对异常值敏感 | 一个离群点可能大幅改变 $r$ |

## 4. 微型数据手算：强力球彩票

### 📖 书中数据

| $x$（头奖） | 334 | 127 | 300 | 227 | 202 | 180 | 164 | 145 | 255 |
|---|---|---|---|---|---|---|---|---|---|
| $y$（销售） | 54 | 16 | 41 | 27 | 23 | 18 | 18 | 16 | 26 |

我们将用公式 10-1 逐步计算 $r$。

In [ ]:
# ========== 步骤 1: 录入数据 ==========
x = np.array([334, 127, 300, 227, 202, 180, 164, 145, 255])
y = np.array([54, 16, 41, 27, 23, 18, 18, 16, 26])
n = len(x)

print(f"📊 步骤 1: 录入数据")
print(f"  n = {n} 对数据")
print(f"  x（头奖/百万$）= {x}")
print(f"  y（销售/百万张）= {y}")
print(f"\n  x̄ = {x.mean():.2f}")
print(f"  ȳ = {y.mean():.2f}")

In [ ]:
# ========== 步骤 2: 计算各分量 ==========
sum_x = np.sum(x)
sum_y = np.sum(y)
sum_xy = np.sum(x * y)
sum_x2 = np.sum(x ** 2)
sum_y2 = np.sum(y ** 2)

print(f"📊 步骤 2: 计算各分量")
print(f"  Σx   = {sum_x}")
print(f"  Σy   = {sum_y}")
print(f"  Σxy  = {sum_xy}")
print(f"  Σx²  = {sum_x2}")
print(f"  Σy²  = {sum_y2}")

In [ ]:
# ========== 步骤 3: 代入公式 10-1 计算 r ==========
numerator = n * sum_xy - sum_x * sum_y
denominator_x = np.sqrt(n * sum_x2 - sum_x ** 2)
denominator_y = np.sqrt(n * sum_y2 - sum_y ** 2)
denominator = denominator_x * denominator_y
r_manual = numerator / denominator

print(f"📊 步骤 3: 代入公式 10-1")
print(f"  分子 = nΣxy - (Σx)(Σy)")
print(f"       = {n}×{sum_xy} - {sum_x}×{sum_y}")
print(f"       = {n * sum_xy} - {sum_x * sum_y}")
print(f"       = {numerator}")
print(f"")
print(f"  √[nΣx² - (Σx)²] = √[{n}×{sum_x2} - {sum_x}²]")
print(f"                   = √[{n * sum_x2} - {sum_x ** 2}]")
print(f"                   = √{n * sum_x2 - sum_x ** 2}")
print(f"                   = {denominator_x:.4f}")
print(f"")
print(f"  √[nΣy² - (Σy)²] = √[{n}×{sum_y2} - {sum_y}²]")
print(f"                   = √[{n * sum_y2} - {sum_y ** 2}]")
print(f"                   = √{n * sum_y2 - sum_y ** 2}")
print(f"                   = {denominator_y:.4f}")
print(f"")
print(f"  r = {numerator} / ({denominator_x:.4f} × {denominator_y:.4f})")
print(f"    = {numerator} / {denominator:.4f}")
print(f"    = {r_manual:.6f}")
print(f"\n💡 手算结果 r ≈ {r_manual:.3f}，与书中给出的 r = 0.947 一致！")
print(f"  强正相关：头奖越高，销售量越大。")

### 📐 用 z 分数版公式 10-2 验证

$$r = \frac{\sum (z_x \cdot z_y)}{n - 1}$$

In [ ]:
# ========== 步骤 4: z 分数版公式验证 ==========
sx = np.std(x, ddof=1)  # 样本标准差
sy = np.std(y, ddof=1)

zx = (x - x.mean()) / sx
zy = (y - y.mean()) / sy

r_zscore = np.sum(zx * zy) / (n - 1)

print(f"📊 步骤 4: z 分数版公式 10-2 验证")
print(f"  sx = {sx:.4f}, sy = {sy:.4f}")
print(f"")
print(f"  zx = {np.round(zx, 4)}")
print(f"  zy = {np.round(zy, 4)}")
print(f"")
print(f"  Σ(zx × zy) = {np.sum(zx * zy):.6f}")
print(f"  r = Σ(zx × zy) / (n-1) = {np.sum(zx * zy):.6f} / {n - 1}")
print(f"    = {r_zscore:.6f}")
print(f"\n  公式 10-1 结果: r = {r_manual:.6f}")
print(f"  公式 10-2 结果: r = {r_zscore:.6f}")
print(f"\n  ✅ 两种公式结果一致！")

## 5. 用 scipy 验证计算结果

### 🔬 scipy.stats.pearsonr 验证

In [ ]:
# ========== 用 scipy 验证 ==========
r_scipy, p_scipy = stats.pearsonr(x, y)

print("🔬 scipy.stats.pearsonr 验证:")
print(f"  手算 r（公式10-1） = {r_manual:.6f}")
print(f"  手算 r（公式10-2） = {r_zscore:.6f}")
print(f"  scipy r            = {r_scipy:.6f}")
print(f"  scipy p 值          = {p_scipy:.6f}")
print(f"\n  ✅ 验证通过！所有方法的 r 值一致。")

In [ ]:
# ========== 强力球数据散点图 ==========
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x, y, color='#2ecc71', s=100, edgecolors='white', linewidths=2, zorder=5)

# 添加回归线
m, b = np.polyfit(x, y, 1)
x_line = np.linspace(x.min() - 10, x.max() + 10, 100)
ax.plot(x_line, m * x_line + b, color='#e74c3c', linewidth=2, linestyle='--',
        label=f'Regression Line (r = {r_scipy:.3f})')

# 标注每个点
for i in range(n):
    ax.annotate(f'({x[i]}, {y[i]})', (x[i], y[i]),
                textcoords='offset points', xytext=(8, 8), fontsize=9)

ax.set_xlabel('Jackpot (Million $)', fontsize=12)
ax.set_ylabel('Sales (Million Tickets)', fontsize=12)
ax.set_title('Powerball Lottery: Jackpot vs Sales', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print(f"  散点图显示头奖金额与销售量之间存在强正线性相关 (r = {r_scipy:.3f})。")
print("  数据点紧密围绕回归线，说明头奖越高，彩票销售量越大。")

## 6. 相关系数的假设检验

即使样本 $r \neq 0$，也可能只是抽样波动造成的。我们需要进行**假设检验**来判断总体相关系数 $\rho$ 是否显著不为 0。

### 📐 假设

$$H_0: \rho = 0 \quad \text{(无线性相关)}$$
$$H_1: \rho \neq 0 \quad \text{(有线性相关)}$$

### 📐 检验统计量（公式 10-3）

$$t = \frac{r \sqrt{n - 2}}{\sqrt{1 - r^2}}, \quad df = n - 2$$

其中：
- $r$ = 样本相关系数
- $n$ = 数据对的数量
- $df = n - 2$ = 自由度

In [ ]:
# ========== 步骤 5: 假设检验 ==========
alpha = 0.05
df = n - 2

t_stat = r_manual * np.sqrt(n - 2) / np.sqrt(1 - r_manual ** 2)
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=df))

print(f"📊 步骤 5: 相关系数的假设检验")
print(f"  H₀: ρ = 0 (无线性相关)")
print(f"  H₁: ρ ≠ 0 (有线性相关)")
print(f"")
print(f"  样本量 n = {n}")
print(f"  自由度 df = {df}")
print(f"  样本 r = {r_manual:.6f}")
print(f"")
print(f"  t = r × √(n-2) / √(1-r²)")
print(f"    = {r_manual:.6f} × √{df} / √(1 - {r_manual:.6f}²)")
print(f"    = {r_manual:.6f} × {np.sqrt(df):.4f} / √{1 - r_manual**2:.6f}")
print(f"    = {r_manual * np.sqrt(df):.4f} / {np.sqrt(1 - r_manual**2):.4f}")
print(f"    = {t_stat:.4f}")
print(f"")
print(f"  p 值 = {p_value:.6f}")
print(f"  显著性水平 α = {alpha}")

if p_value < alpha:
    print(f"\n  💡 结论: p < α，拒绝 H₀")
    print(f"  头奖金额与销售量之间存在显著的线性相关。")
else:
    print(f"\n  💡 结论: p ≥ α，不拒绝 H₀")
    print(f"  没有足够证据说明两者之间存在线性相关。")

In [ ]:
# ========== 用 scipy 验证假设检验 ==========
print("🔬 scipy 验证:")
print(f"  手算 t 统计量 = {t_stat:.6f}")
print(f"  scipy r        = {r_scipy:.6f}")
print(f"  scipy p 值     = {p_scipy:.6f}")
print(f"  手算 p 值      = {p_value:.6f}")

# pearsonr 返回的 p 值与手动计算一致
print(f"\n  ✅ 验证通过！p 值 = {p_scipy:.6f} < 0.05，拒绝 H₀。")

In [ ]:
# ========== t 分布与检验统计量位置 ==========
fig, ax = plt.subplots(figsize=(10, 6))

x_t = np.linspace(-5, 5, 1000)
y_t = stats.t.pdf(x_t, df=df)
ax.plot(x_t, y_t, 'k-', linewidth=2, label=f't Distribution (df={df})')

# 标记临界值
t_critical = stats.t.ppf(1 - alpha / 2, df=df)
ax.axvline(x=-t_critical, color='#e74c3c', linestyle='--', alpha=0.7,
           label=f'Critical Value = ±{t_critical:.3f}')
ax.axvline(x=t_critical, color='#e74c3c', linestyle='--', alpha=0.7)

# 填充拒绝域
x_reject_left = np.linspace(-5, -t_critical, 200)
x_reject_right = np.linspace(t_critical, 5, 200)
ax.fill_between(x_reject_left, 0, stats.t.pdf(x_reject_left, df=df),
                alpha=0.3, color='#e74c3c', label='Rejection Region')
ax.fill_between(x_reject_right, 0, stats.t.pdf(x_reject_right, df=df),
                alpha=0.3, color='#e74c3c')

# 标记 t 统计量位置
ax.axvline(x=t_stat, color='#2ecc71', linewidth=2, linestyle='-',
           label=f't statistic = {t_stat:.3f}')

ax.set_xlabel('T Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Hypothesis Test for Correlation Coefficient', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  t 统计量 = {t_stat:.3f} 远远超出临界值 ±{t_critical:.3f}，落入拒绝域。")
print(f"  p 值 = {p_value:.6f} < α = 0.05，拒绝 H₀。")
print(f"  结论：头奖金额与销售量之间存在显著的线性相关。")

## 7. 相关系数的解读

### 📊 不同 $r$ 值的含义

| $|r|$ 范围 | 相关强度 | 说明 |
|---|---|---|
| 0.00 – 0.19 | 很弱 | 几乎没有线性关系 |
| 0.20 – 0.39 | 弱 | 线性关系不明显 |
| 0.40 – 0.59 | 中等 | 有一定线性关系 |
| 0.60 – 0.79 | 强 | 线性关系较明显 |
| 0.80 – 1.00 | 很强 | 线性关系非常显著 |

> ⚠️ 以上分段仅供参考，实际解读需结合具体领域。在物理学中 $r = 0.9$ 可能算"一般"，而在社会科学中 $r = 0.5$ 已算"很强"。

In [ ]:
# ========== 不同 r 值的散点图对比 ==========
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

r_targets = [0.0, 0.3, 0.5, -0.5, -0.8, 0.95]
r_labels = ['r ≈ 0.0', 'r ≈ 0.3', 'r ≈ 0.5', 'r ≈ -0.5', 'r ≈ -0.8', 'r ≈ 0.95']
colors = ['steelblue', '#e67e22', '#e67e22', '#e74c3c', '#e74c3c', '#2ecc71']

for idx, (r_target, label, color) in enumerate(zip(r_targets, r_labels, colors)):
    ax = axes[idx // 3, idx % 3]
    n_pts = 80
    x_pts = np.random.normal(0, 1, n_pts)
    noise_std = np.sqrt(1 - r_target ** 2) / abs(r_target) if abs(r_target) > 0.05 else 1
    y_pts = r_target * x_pts + np.random.normal(0, noise_std, n_pts)
    # Rescale to match target r more closely
    r_actual, _ = stats.pearsonr(x_pts, y_pts)
    ax.scatter(x_pts, y_pts, color=color, alpha=0.6, edgecolors='white', s=50)
    ax.set_title(f'{label} (actual: {r_actual:.2f})', fontsize=13, fontweight='bold')
    ax.set_xlabel('X', fontsize=11)
    ax.set_ylabel('Y', fontsize=11)
    ax.grid(alpha=0.3)

plt.suptitle('Correlation Coefficient Examples', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  r = 0.0：点完全随机散布，无线性关系")
print("  r = 0.3：弱正相关，趋势不明显")
print("  r = 0.5：中等正相关，趋势开始显现")
print("  r = -0.5：中等负相关，x 增大时 y 有减小趋势")
print("  r = -0.8：强负相关，点较紧密围绕下降直线")
print("  r = 0.95：很强正相关，点非常紧密围绕上升直线")

## 8. 相关性 ≠ 因果性

### 📖 书中要点
> **即使 $r$ 很大且检验显著，也不能据此断定 $x$ 与 $y$ 之间存在因果关系。**

### 💡 为什么？

相关系数 $r$ 只能告诉我们两个变量**是否一起变化**，但不能告诉我们**为什么**一起变化。

可能存在以下情况：

| 情况 | 例子 |
|------|------|
| $x$ 导致 $y$ | 吸烟 → 肺癌风险 |
| $y$ 导致 $x$ | 因果方向反过来 |
| 第三变量 $z$ 同时影响 $x$ 和 $y$ | 冰淇淋销量 & 溺水人数（$z$ = 气温） |
| 纯属巧合 | 越狱电影数量 & 采矿工程学位授予数 |

### 🔗 "相关不蕴涵因果"的经典案例

「冰淇淋销量」与「溺水死亡人数」高度正相关——但这不是因为吃冰淇淋导致溺水！
真正的原因是**气温**（混杂变量）：气温升高 → 冰淇淋销量增加，气温升高 → 游泳人数增加 → 溺水人数增加。

In [ ]:
# ========== 经典案例：冰淇淋 vs 溺水（混杂变量演示） ==========
n_days = 100

# 混杂变量：气温
temperature = np.random.uniform(15, 38, n_days)

# 冰淇淋销量 = f(气温) + 噪声
ice_cream_sales = 2 * temperature + np.random.normal(0, 8, n_days)

# 溺水人数 = f(气温) + 噪声
drowning_deaths = 0.5 * temperature + np.random.normal(0, 3, n_days)

r_ice_drown, p_ice_drown = stats.pearsonr(ice_cream_sales, drowning_deaths)
r_ice_temp, _ = stats.pearsonr(ice_cream_sales, temperature)
r_drown_temp, _ = stats.pearsonr(drowning_deaths, temperature)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 冰淇淋 vs 溺水
axes[0].scatter(ice_cream_sales, drowning_deaths, color='#e74c3c', alpha=0.6,
                edgecolors='white', s=50)
axes[0].set_xlabel('Ice Cream Sales', fontsize=12)
axes[0].set_ylabel('Drowning Deaths', fontsize=12)
axes[0].set_title(f'Ice Cream vs Drowning: r = {r_ice_drown:.2f}',
                  fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)

# 冰淇淋 vs 气温
axes[1].scatter(temperature, ice_cream_sales, color='#2ecc71', alpha=0.6,
                edgecolors='white', s=50)
axes[1].set_xlabel('Temperature (°C)', fontsize=12)
axes[1].set_ylabel('Ice Cream Sales', fontsize=12)
axes[1].set_title(f'Temp vs Ice Cream: r = {r_ice_temp:.2f}',
                  fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)

# 溺水 vs 气温
axes[2].scatter(temperature, drowning_deaths, color='steelblue', alpha=0.6,
                edgecolors='white', s=50)
axes[2].set_xlabel('Temperature (°C)', fontsize=12)
axes[2].set_ylabel('Drowning Deaths', fontsize=12)
axes[2].set_title(f'Temp vs Drowning: r = {r_drown_temp:.2f}',
                  fontsize=13, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.suptitle('Correlation ≠ Causation: The Confounding Variable Problem',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print(f"  冰淇淋销量 vs 溺水人数: r = {r_ice_drown:.2f}（看似高度相关！）")
print(f"  但气温 vs 冰淇淋销量:   r = {r_ice_temp:.2f}")
print(f"  且气温 vs 溺水人数:     r = {r_drown_temp:.2f}")
print(f"\n  ⚠️ 真正的因果链：气温 → 冰淇淋销量 ↑ & 溺水人数 ↑")
print(f'  冰淇淋和溺水只是「碰巧」都随气温变化，并没有因果关系。')
print(f'  这就是「混杂变量」(confounding variable)导致的「伪相关」。')

## 9. 可解释变异 $r^2$

### 📐 可解释变异（Coefficient of Determination）

$$r^2 = \text{可解释变异比例}$$

其中：
- $r^2$ 表示因变量 $y$ 的变异中，可以由自变量 $x$ 的线性关系解释的比例
- $1 - r^2$ 表示无法由线性关系解释的残余变异

### 💡 例如
- 若 $r = 0.947$，则 $r^2 = 0.897$
- 含义：$y$ 的变异中有 89.7% 可以由 $x$ 的线性关系解释
- 剩余 10.3% 来自其他未知因素或随机波动

In [ ]:
# ========== 可解释变异计算 ==========
r2 = r_manual ** 2
unexplained = 1 - r2

print("📊 可解释变异分析")
print(f"  r  = {r_manual:.6f}")
print(f"  r² = {r2:.6f}")
print(f"")
print(f"  可解释变异: {r2*100:.2f}%")
print(f"  残余变异:   {unexplained*100:.2f}%")
print(f"")
print(f"  💡 含义：头奖金额的变化可以解释销售量变化的 {r2*100:.1f}%。")
print(f"  剩余 {unexplained*100:.1f}% 的变异来自其他因素（如媒体宣传、节假日等）。")

# 可视化可解释变异
fig, ax = plt.subplots(figsize=(8, 5))
labels = ['Explained\n(r²)', 'Unexplained\n(1 - r²)']
sizes = [r2, unexplained]
colors_pie = ['#2ecc71', '#e74c3c']
wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors_pie,
                                   autopct='%1.1f%%', startangle=90,
                                   textprops={'fontsize': 12})
for autotext in autotexts:
    autotext.set_fontsize(14)
    autotext.set_fontweight('bold')
ax.set_title(f'Explained vs Unexplained Variance (r = {r_manual:.3f})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print(f"  绿色区域（{r2*100:.1f}%）= 可由线性关系解释的变异")
print(f"  红色区域（{unexplained*100:.1f}%）= 残余变异（其他因素或随机波动）")

In [ ]:
# ========== 不同 r 值对应的 r² ==========
r_values = np.array([0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99])
r2_values = r_values ** 2

print("📊 不同 r 值对应的可解释变异 r²：")
print(f"{'r':>8} {'r²':>8} {'可解释%':>10} {'残余%':>10}")
print("-" * 40)
for r_val, r2_val in zip(r_values, r2_values):
    print(f"{r_val:>8.2f} {r2_val:>8.4f} {r2_val*100:>9.1f}% {(1-r2_val)*100:>9.1f}%")

print("\n💡 关键洞察：")
print("  r = 0.7 时，r² = 0.49，即只有 49% 的变异被解释")
print("  r = 0.9 时，r² = 0.81，即 81% 的变异被解释")
print("  r 从 0.9 提升到 0.99，r² 从 0.81 提升到 0.98——边际收益递减！")

## 10. 大样本扩展

微型企业数据只有 9 对，样本量太小。让我们用模拟数据验证在更大样本下 $r$ 的行为。

### 📐 数据生成过程
- $x$：正态分布 $N(100, 30)$
- $y$：$y = 0.8x + \varepsilon$，其中 $\varepsilon \sim N(0, 20)$
- 样本量：$n = 300$

In [ ]:
# ========== 大样本模拟 ==========
n_large = 300

# 数据生成过程
x_large = np.random.normal(100, 30, n_large)
y_large = 0.8 * x_large + np.random.normal(0, 20, n_large)

# 计算相关系数
r_large, p_large = stats.pearsonr(x_large, y_large)
r2_large = r_large ** 2

print(f"📊 大样本模拟结果")
print(f"  样本量 n = {n_large}")
print(f"  x ~ N(100, 30)")
print(f"  y = 0.8x + ε, ε ~ N(0, 20)")
print(f"")
print(f"  r  = {r_large:.6f}")
print(f"  r² = {r2_large:.6f}")
print(f"  p  = {p_large:.6e}")
print(f"")
print(f"  可解释变异: {r2_large*100:.2f}%")
print(f"  残余变异:   {(1-r2_large)*100:.2f}%")

In [ ]:
# ========== 大样本散点图 ==========
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x_large, y_large, color='steelblue', alpha=0.4, edgecolors='white',
           s=40, label=f'n = {n_large}')

# 回归线
m_l, b_l = np.polyfit(x_large, y_large, 1)
x_line_l = np.linspace(x_large.min(), x_large.max(), 100)
ax.plot(x_line_l, m_l * x_line_l + b_l, color='#e74c3c', linewidth=2,
        linestyle='--', label=f'Fit: y = {m_l:.2f}x + {b_l:.2f}')

ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('Y', fontsize=12)
ax.set_title(f'Large Sample Correlation (n={n_large}, r={r_large:.3f})',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  大样本 (n={n_large}) 下，r = {r_large:.3f}，p 值极小（{p_large:.2e}）。")
print(f"  样本量越大，检验的统计功效越强，越容易检测到微弱的相关。")
print(f"  ⚠️ 但 r 的大小不随 n 增大而增大——n 大只影响检验的灵敏度。")

## 11. 异常值对 $r$ 的影响

### 📖 书中要点
> Pearson 相关系数对异常值非常敏感。一个极端的离群点可能大幅改变 $r$ 的值。

下面我们用一个例子来演示。

In [ ]:
# ========== 异常值对 r 的影响 ==========
# 原始数据（无异常值）
x_clean = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
y_clean = np.array([2, 4, 5, 4, 6, 7, 8, 7, 9, 10])
r_clean, _ = stats.pearsonr(x_clean, y_clean)

# 添加一个异常值
x_outlier = np.append(x_clean, 15)
y_outlier = np.append(y_clean, 1)  # 极端低值
r_outlier, _ = stats.pearsonr(x_outlier, y_outlier)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 无异常值
axes[0].scatter(x_clean, y_clean, color='#2ecc71', s=80, edgecolors='white', zorder=5)
m_c, b_c = np.polyfit(x_clean, y_clean, 1)
axes[0].plot(x_clean, m_c * x_clean + b_c, 'r--', linewidth=2)
axes[0].set_title(f'No Outlier: r = {r_clean:.3f}', fontsize=14, fontweight='bold')
axes[0].set_xlabel('X', fontsize=12)
axes[0].set_ylabel('Y', fontsize=12)
axes[0].grid(alpha=0.3)

# 有异常值
axes[1].scatter(x_outlier[:-1], y_outlier[:-1], color='#2ecc71', s=80,
                edgecolors='white', zorder=5, label='Normal')
axes[1].scatter(x_outlier[-1], y_outlier[-1], color='#e74c3c', s=150,
                edgecolors='white', zorder=6, marker='*', label='Outlier')
m_o, b_o = np.polyfit(x_outlier, y_outlier, 1)
axes[1].plot(x_outlier, m_o * x_outlier + b_o, 'r--', linewidth=2)
axes[1].set_title(f'With Outlier: r = {r_outlier:.3f}', fontsize=14, fontweight='bold')
axes[1].set_xlabel('X', fontsize=12)
axes[1].set_ylabel('Y', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.suptitle('Impact of Outlier on Correlation Coefficient',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print(f"  无异常值时 r = {r_clean:.3f}（强正相关）")
print(f"  加入一个异常值后 r = {r_outlier:.3f}（大幅下降！）")
print(f"  一个离群点就让 r 从 {r_clean:.2f} 变为 {r_outlier:.2f}，变化了 {abs(r_clean - r_outlier):.2f}。")
print(f"  ⚠️ 这就是为什么在计算 r 之前，一定要先画散点图检查异常值。")

## 📌 核心概念回顾

### 📌 Pearson 相关系数 (Pearson Correlation Coefficient)
- **定义**: 度量两个变量之间**线性关系强度和方向**的数值指标
- **公式**: $r = \dfrac{n\sum xy - (\sum x)(\sum y)}{\sqrt{n\sum x^2 - (\sum x)^2} \cdot \sqrt{n\sum y^2 - (\sum y)^2}}$
- **含义/意义**: $r > 0$ 表示正相关，$r < 0$ 表示负相关，$|r|$ 越接近 1 关系越强
- **判断标准**: $|r| < 0.2$ 很弱，$0.2-0.4$ 弱，$0.4-0.6$ 中等，$0.6-0.8$ 强，$> 0.8$ 很强

### 📌 可解释变异 (Coefficient of Determination)
- **定义**: $r^2$ 表示因变量的变异中可由自变量线性解释的比例
- **公式**: $r^2 = (\text{Pearson } r)^2$
- **含义/意义**: $r^2 = 0.81$ 意味着 81% 的变异可解释，19% 为残余
- **判断标准**: 越接近 1，模型解释力越强

### 📌 假设检验 (Hypothesis Test for $\rho$)
- **定义**: 检验总体相关系数 $\rho$ 是否显著不为 0
- **公式**: $t = \dfrac{r\sqrt{n-2}}{\sqrt{1-r^2}}$，$df = n-2$
- **含义/意义**: $p < \alpha$ 说明样本 $r$ 不太可能来自 $\rho = 0$ 的总体
- **判断标准**: $p < 0.05$ 拒绝 $H_0$，认为存在显著线性相关

### 📌 相关性 vs 因果性 (Correlation vs Causation)
- **定义**: 相关系数只能度量共变关系，不能证明因果方向
- **公式**: 无
- **含义/意义**: 高度相关可能来自因果、反向因果、混杂变量或巧合
- **判断标准**: 需要随机对照实验（RCT）或其他因果推断方法才能确定因果

### 🔗 完整流程
```
画散点图 → 计算 r → 计算 r² → 假设检验 → 解读
    ↓          ↓         ↓          ↓         ↓
  直观判断   线性强度   可解释比例  p 值判断   领域意义
```

### 📝 检验指标汇总

| 指标 | 公式 | 含义 | 取值范围 |
|------|------|------|----------|
| $r$ | 公式 10-1 / 10-2 | 线性相关强度和方向 | $[-1, 1]$ |
| $r^2$ | $r^2$ | 可解释变异比例 | $[0, 1]$ |
| $t$ | $t = r\sqrt{n-2}/\sqrt{1-r^2}$ | 检验统计量 | $(-\infty, +\infty)$ |
| $p$ | $P(|T| > |t|)$ | 显著性概率 | $[0, 1]$ |

## ❌ 常见误区

### ❌ 误区 1：相关系数高就意味着因果关系
**✓ 正确理解**: 相关系数只能度量两个变量的**共变关系**，不能证明因果方向。高度相关可能是由混杂变量、反向因果或纯属巧合造成的。要确定因果关系，需要随机对照实验或其他因果推断方法。

### ❌ 误区 2：$r = 0$ 就说明两个变量没有关系
**✓ 正确理解**: $r = 0$ 只说明没有**线性**关系，但可能存在非线性关系（如抛物线、周期性等）。$r$ 只捕捉线性模式，画散点图才能发现非线性模式。

### ❌ 误区 3：$r$ 的大小可以跨领域直接比较
**✓ 正确理解**: 不同领域的"强相关"标准不同。物理学中 $r = 0.9$ 可能算"一般"，社会科学中 $r = 0.5$ 已算"很强"。解读 $r$ 时必须结合具体背景和研究领域。

### ❌ 误区 4：只要 $r$ 大就可以用线性模型预测
**✓ 正确理解**: $r$ 大只说明线性关系强，但还需要检查：(1) 散点图是否呈线性形态；(2) 是否有异常值扭曲了 $r$；(3) 数据范围是否足够宽（范围受限会低估 $r$）；(4) 样本是否具有代表性。

### ❌ 误区 5：$r$ 不受变量单位影响，所以可以直接比较不同数据集的 $r$
**✓ 正确理解**: 虽然 $r$ 本身不受量纲影响，但**数据的变异范围**会影响 $r$。如果某个数据集的 $x$ 范围很窄，$r$ 可能被人为压低。比较不同数据集的 $r$ 时，需确保数据范围和采样方式可比。